In [2]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="gemma3:4b")

## Pydentic

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The moveies rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatOllama(output_version=None, model='gemma3:4b'), kwargs={'format': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'rating': {'description': 'The moveies rating out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [6]:
response=model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [7]:
type(response)

__main__.Movie

### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A Movie with details"""
    
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The moveies rating out of 10")


model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")

response

{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}\n\n \t \t \t \t \t \t \t \t \t \t', additional_kwargs={}, response_metadata={'model': 'gemma3:4b', 'created_at': '2026-05-22T09:17:48.307975Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8327391833, 'load_duration': 281986458, 'prompt_eval_count': 16, 'prompt_eval_duration': 1567218958, 'eval_count': 53, 'eval_duration': 5405368206, 'logprobs': None, 'model_name': 'gemma3:4b', 'model_provider': 'ollama'}, id='lc_run--019e4ef9-e2f6-7493-82fc-630ff0acbc0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 53, 'total_tokens': 69}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

In [10]:
### Nested structure

In [13]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

In [15]:
print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Ken Watanabe', role='Yinsen')] genres=['Action', 'Sci-Fi', 'Thriller'] budget=160000000.0


## TypeDict

In [18]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]

model_with_type_dict=model.with_structured_output(MovieDict)
response=model_with_type_dict.invoke("Please provide the details of the movie avengers")
response

{'title': 'Avengers: Endgame',
 'year': 2019,
 'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4}

In [20]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Marion Cotillard', 'role': 'Mal'},
  {'name': 'Ken Watanabe', 'role': 'Yinsen'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'budget': 160000000.0}

## DataClasses

In [27]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
## Pydentic

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str =Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone:int = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='6ad8792b-8252-4667-95b7-7ac0109647c9'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":5551234567}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 608, 'prompt_tokens': 204, 'total_tokens': 812, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DiGuV0vjsbzhndPnx13afmLNvQVO7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4f1d-b0c9-7903-9025-740fd186cd70-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens'

In [34]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone=5551234567


In [40]:
## Typedict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str 
    email: str
    phone:int

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='ac0f1fe6-7812-42fe-8520-b60acc80d35c'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":5551234567}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 544, 'prompt_tokens': 178, 'total_tokens': 722, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DiGyV4GsX4LaF5f581G3IGz8YmGDR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4f21-7bae-7ca0-a50c-b257394f3da6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 178, 'output_tokens'

In [41]:
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': 5551234567}

In [42]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str 
    email: str
    phone:int

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone=5551234567)
